# Stage 2 — Instruction Fine-Tuning with LoRA

**Starts from:** the domain-adapted model produced by `01_domain_adaptation_lora.ipynb`
**Hardware:** free-tier Colab T4
**Runtime:** ~10 minutes end to end

---

## What this notebook does

Notebook 01 left you with a model that writes fluent Meridian Trust documentation and **does not
answer questions**. This notebook fixes that by training a second LoRA on paired
instruction/input/output examples.

The pipeline is the one real fine-tuning programmes use:

```
pretrained base
      │
      ├─ stage 1: domain adaptation on raw text     → LoRA #1
      │
      ├─ MERGE LoRA #1 into the base weights        → clean domain-adapted base
      │
      └─ stage 2: instruction tuning on pairs       → LoRA #2
```

**Why merge instead of stacking two adapters?** Stacked adapters interact in ways that are hard
to predict and harder to validate. Merging produces a single clean base for stage 2, and it means
the stage-2 adapter can be reasoned about on its own.

## The three things here that most tutorials get wrong

| | |
|---|---|
| **Loading the stage-1 adapter** | `AutoModelForCausalLM.from_pretrained(lora_dir)` **silently ignores the adapter** and gives you the base model back. You need `PeftModel.from_pretrained`. We assert the weights actually changed. |
| **Completion-only loss** | Supervising the whole sequence teaches the model to generate `### Instruction:` headers. The loss belongs on the response only. |
| **EOS on targets** | Omit it and the model answers correctly, then keeps going forever — inventing further questions and answering those too. |

**Set your runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.

## 1. Runtime and dependencies

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "No GPU found — set Runtime > Change runtime type > T4 GPU")

In [ ]:
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model(). Nothing here uses torchao, so remove it rather than chase a
# compatible build against Colab's torch.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

In [ ]:
import gc, json, math, random
from pathlib import Path

import torch
import transformers
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU."

# bfloat16 needs Ampere (sm_80) or later; a T4 is Turing (sm_75). Checking compute
# capability directly, because torch.cuda.is_bf16_supported() has returned True on Turing.
SUPPORTS_BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"

MODEL_ID = "unsloth/Llama-3.2-1B"

print(f"transformers {transformers.__version__} | GPU {torch.cuda.get_device_name(0)}")
print(f"compute dtype {DTYPE} | from_pretrained keyword {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Srikesh2197/fine_tuning.git"
CLONE_DIR = Path("/content/fine_tuning")
SENTINEL = Path("data/instruction/train.jsonl")


def find_repo() -> Path:
    """Locate the repo root: local checkout, public clone, or Google Drive copy."""
    # 1. Already inside a checkout (local Jupyter, or a clone from earlier this session).
    for candidate in [Path.cwd(), *Path.cwd().parents, CLONE_DIR]:
        if (candidate / SENTINEL).exists():
            return candidate

    # 2. Plain clone. Works if the repo is public. Fails fast and harmlessly if it is
    #    private, because a Colab runtime carries no GitHub credentials.
    print(f"Trying to clone {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
                   capture_output=True, text=True)
    if (CLONE_DIR / SENTINEL).exists():
        print("  cloned")
        return CLONE_DIR
    print("  clone failed (expected while the repo is private) - checking Google Drive")
    # 3. A copy in Google Drive. This is the private-repo path: put the repo folder in
    #    MyDrive once, and every future session finds it with no credentials. We search
    #    by content rather than by folder name, so whatever you called it works.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        for folder in sorted(drive_root.iterdir()):
            try:
                if folder.is_dir() and (folder / SENTINEL).exists():
                    print(f"  found repo in Drive: {folder}")
                    return folder
            except OSError:
                continue

    raise RuntimeError(
        "Could not find the repo data. Two ways to fix this:\n"
        f"  (a) Make {REPO_URL} public, then re-run this cell; or\n"
        "  (b) keep it private and upload the repo folder into the top level of your\n"
        f"      Google Drive, so that MyDrive/<folder>/{SENTINEL} exists,\n"
        "      then re-run this cell."
    )


REPO = find_repo()
print(f"Repo root: {REPO}")

## 2. Find the stage-1 adapter

Notebook 01 saved it to Drive at `MyDrive/finetuning-demo/stage1-domain-lora`. If you skipped
notebook 01, set `SKIP_STAGE1 = True` below and this notebook will instruction-tune the raw base
model instead — useful for comparison, but you lose half the story.

In [ ]:
SKIP_STAGE1 = False

STAGE1_DIR = None
if not SKIP_STAGE1:
    candidates = [
        Path("/content/drive/MyDrive/finetuning-demo/stage1-domain-lora"),
        Path("/content/outputs/stage1-domain-lora"),
    ]
    if not any(c.exists() for c in candidates):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except ImportError:
            pass
    STAGE1_DIR = next((c for c in candidates if (c / "adapter_config.json").exists()), None)

if STAGE1_DIR is None:
    print("No stage-1 adapter found. Either run notebook 01 first, or set SKIP_STAGE1 = True.")
else:
    print(f"Found stage-1 adapter: {STAGE1_DIR}\n")
    cfg = json.loads((STAGE1_DIR / "adapter_config.json").read_text())
    for key in ["base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules"]:
        print(f"  {key:26s} {cfg.get(key)}")

## 3. Rebuild the domain-adapted base by merging

This is the step to get right. Three sub-steps:

1. Load the **pristine** base model.
2. Attach the stage-1 adapter with `PeftModel.from_pretrained` — *not*
   `AutoModelForCausalLM.from_pretrained(adapter_dir)`, which loads a base model and quietly
   ignores the adapter files sitting next to it.
3. `merge_and_unload()` to fold `(α/r)·BA` into `W`, leaving a plain `LlamaForCausalLM` with no
   PEFT wrapper — a clean base for stage 2.

We snapshot a weight tensor before and after so the merge is **verified**, not assumed. This is
the check that would have caught the bug in the reference notebooks.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"          # training, not generation
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
model.config.pad_token_id = tokenizer.pad_token_id

# Snapshot a weight the stage-1 adapter targeted, so we can prove the merge changed it.
PROBE_WEIGHT = "model.layers.8.mlp.down_proj.weight"
before = dict(model.named_parameters())[PROBE_WEIGHT].detach().clone()
print(f"Loaded pristine {MODEL_ID}")

In [ ]:
if STAGE1_DIR is not None:
    model = PeftModel.from_pretrained(model, STAGE1_DIR)
    print(f"Attached stage-1 adapter ({type(model).__name__})")
    model = model.merge_and_unload()
    print(f"Merged and unwrapped   ({type(model).__name__})")

    after = dict(model.named_parameters())[PROBE_WEIGHT].detach()
    drift = (after.float() - before.float()).abs().max().item()
    changed = not torch.equal(after, before)

    print(f"\nVerification on {PROBE_WEIGHT}")
    print(f"  weights changed : {changed}")
    print(f"  max |delta|     : {drift:.6f}")
    assert changed, (
        "The merge did not change any weights. The adapter was not applied — this is exactly "
        "the failure mode that AutoModelForCausalLM.from_pretrained(adapter_dir) produces."
    )
    print("\n  -> stage-1 knowledge is now inside the base weights")
else:
    print("SKIP_STAGE1 is set: instruction-tuning the raw base model.")

del before
gc.collect(); torch.cuda.empty_cache()
print(f"\nGPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. The synthetic instruction data

The dataset is built by `scripts/generate_instructions.py`. Its structure, and an honest account
of what "synthetic" means here, is in `data/instruction/README.md`. In short: ~120 QA seeds are
hand-authored against the domain corpus, then expanded programmatically into surface variants,
plus classification tasks derived from document structure and a set of **abstention** examples.

Regenerating is optional — the output is committed and is byte-identical for a given seed.

In [ ]:
# Optional: regenerate to see the pipeline run. Output is deterministic.
!cd {REPO} && python scripts/generate_instructions.py

In [ ]:
def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]


train_records = read_jsonl(REPO / "data" / "instruction" / "train.jsonl")
eval_records = read_jsonl(REPO / "data" / "instruction" / "eval.jsonl")

from collections import Counter
print(f"train {len(train_records)}   eval {len(eval_records)}\n")
print("task kinds in train:", dict(Counter(r["kind"] for r in train_records)))
print("task kinds in eval :", dict(Counter(r["kind"] for r in eval_records)))
print(f"\nrecords with a non-empty `input`: {sum(1 for r in train_records if r['input'])}")

In [ ]:
for kind in ["qa", "excerpt", "subject", "abstain"]:
    r = next(x for x in train_records if x["kind"] == kind)
    print("=" * 96)
    print(f"[{kind}]")
    print(f"  instruction : {r['instruction']}")
    print(f"  input       : {(r['input'][:120] + '...') if r['input'] else '(empty)'}")
    print(f"  output      : {r['output'][:200]}")
print("=" * 96)

### Why there are abstention examples in here

15 of these records answer "the documentation does not cover that". If every training example has
a confident answer, the model learns that every question has one — and then answers confidently
when it should decline. That is the most damaging failure mode for anything retrieval-adjacent,
and it is installed by the *absence* of abstention data rather than by anything you did wrong.

## 5. The prompt template

`unsloth/Llama-3.2-1B` is a **base** model — it ships with no chat template, so we define our own.
We use the Alpaca format, with two variants depending on whether `input` is populated.

**The EOS token is not decoration.** It is the only signal that teaches the model where a response
ends. Leave it off and you get a model that answers correctly and then keeps generating —
inventing follow-up questions and answering those. That symptom is baffling until you've seen it
once, and no decoding parameter can repair it.

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction describing a task, paired with input providing further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_NO_INPUT = (
    "Below is an instruction describing a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def build_prompt(record) -> str:
    """The part the model reads but is NOT trained to produce."""
    template = PROMPT_WITH_INPUT if record["input"].strip() else PROMPT_NO_INPUT
    return template.format(instruction=record["instruction"].strip(),
                           input=record["input"].strip())


example = train_records[0]
print(build_prompt(example) + example["output"] + tokenizer.eos_token)

## 6. Completion-only loss masking

This is the highest-leverage cell in the notebook.

The naive approach sets `labels = input_ids.copy()`, which supervises **every** token — including
the template boilerplate and the user's own question. The model dutifully learns to generate
`### Instruction:` headers and plausible questions, because that is what it was trained to do.

Instead, we tokenize the prompt and the full sequence separately, then set `labels = -100` across
the prompt span. `-100` is PyTorch's `ignore_index`: those positions contribute nothing to the
loss. The model still *reads* the prompt (it's in `input_ids`), it just isn't *scored* on it.

```
input_ids :  [Below is an instruction ... ### Response:\n]  [answer tokens]  [EOS]
labels    :  [-100 -100 -100 -100 ...            -100  ]  [answer tokens]  [EOS]
             └──────────── read, not scored ──────────┘   └── supervised ──┘
```

In [ ]:
MAX_LEN = 512


def tokenize_record(record):
    """Tokenize one record, masking the prompt out of the loss."""
    prompt = build_prompt(record)
    answer = record["output"].strip()

    # add_special_tokens=True puts BOS at the front of the prompt only.
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_ids = answer_ids + [tokenizer.eos_token_id]   # <-- teaches the model to stop

    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    # -100 over the prompt span, real ids over the answer span.
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


train_tok = [tokenize_record(r) for r in train_records]
eval_tok = [tokenize_record(r) for r in eval_records]

supervised = sum(sum(1 for t in r["labels"] if t != -100) for r in train_tok)
total = sum(len(r["labels"]) for r in train_tok)
lengths = sorted(len(r["input_ids"]) for r in train_tok)

print(f"sequence length: p50={lengths[len(lengths)//2]} p95={lengths[int(len(lengths)*.95)]} max={lengths[-1]}")
print(f"supervised tokens: {supervised:,} / {total:,} = {supervised/total:.1%}")
print("\nThe rest is prompt the model reads but is not scored on. If this figure were ~100%,")
print("the prompt mask would not be working.")

truncated = sum(1 for r in train_tok if len(r["input_ids"]) >= MAX_LEN)
assert truncated == 0, f"{truncated} records hit MAX_LEN — a truncated target teaches the model to stop mid-sentence"
print(f"records truncated at MAX_LEN={MAX_LEN}: {truncated}")

### Sanity check: show exactly which tokens are supervised

Don't take the masking on trust. Decode it.

In [ ]:
sample = train_tok[0]
kept = [t for t in sample["labels"] if t != -100]
masked_n = len(sample["labels"]) - len(kept)

print(f"total tokens    : {len(sample['input_ids'])}")
print(f"masked (-100)   : {masked_n}")
print(f"supervised      : {len(kept)}\n")
print("--- MASKED (read, not scored) " + "-" * 50)
print(tokenizer.decode(sample["input_ids"][:masked_n]))
print("\n--- SUPERVISED (the training signal) " + "-" * 43)
print(tokenizer.decode(kept))
print("\nNote the trailing <|end_of_text|>: that is the EOS the model learns to emit.")

## 7. Collator

Sequences here are variable length, so we pad per batch (dynamic padding) rather than to a global
maximum — less wasted compute than padding everything to 512.

The important detail: `input_ids` pads with the pad token, `attention_mask` pads with `0`, and
`labels` pad with **`-100`**. Padding labels with the pad token id instead would put the model
right back to training on padding.

In [ ]:
from datasets import Dataset


def collate(features):
    """Dynamic padding. Note the three different pad values."""
    longest = max(len(f["input_ids"]) for f in features)

    def pad(seq, value):
        return seq + [value] * (longest - len(seq))

    return {
        "input_ids": torch.tensor([pad(f["input_ids"], tokenizer.pad_token_id) for f in features]),
        "attention_mask": torch.tensor([pad(f["attention_mask"], 0) for f in features]),
        "labels": torch.tensor([pad(f["labels"], -100) for f in features]),
    }


train_ds = Dataset.from_list(train_tok)
eval_ds = Dataset.from_list(eval_tok)

demo = collate([train_tok[0], train_tok[1]])
print({k: tuple(v.shape) for k, v in demo.items()})
print(f"\nlabel pad value in row 0, last position: {demo['labels'][0][-1].item()}  (should be -100 if padded)")

## 8. Attach a fresh LoRA and train

A **new** adapter, on the merged domain-adapted base. Stage 1's knowledge is in the base weights
now, so this adapter only has to learn one thing: how to respond when addressed.

`r=16` again. Instruction-following is largely a behavioural change and would probably survive
`r=8`; keeping it at 16 makes the two stages directly comparable.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Same fp16 gotcha as stage 1: trainable params must be fp32 or the grad scaler refuses to
# unscale them. Base weights stay in fp16 — that's where the memory is.
for _, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()

model.config.use_cache = False

In [ ]:
from transformers import Trainer, TrainingArguments

OUT_DIR = "/content/outputs/stage2-instruct-lora"

args = TrainingArguments(
    output_dir=OUT_DIR,
    # note: no overwrite_output_dir - transformers v5 removed it
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,      # effective batch 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate,
)

steps = len(train_ds) // (args.per_device_train_batch_size * args.gradient_accumulation_steps)
print(f"optimizer steps: ~{steps * args.num_train_epochs}")
trainer.train()

## 9. Loss curves

These losses are **not comparable to notebook 01's**. Stage 1 scored every token in a packed
block; stage 2 scores only response tokens. Different denominators, different tasks.

In [ ]:
history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*train_pts), label="train loss", color="#4C72B0", alpha=0.8)
if eval_pts:
    plt.plot(*zip(*eval_pts), label="eval loss (held-out instructions)",
             color="#C44E52", marker="o")
plt.xlabel("epoch"); plt.ylabel("loss (response tokens only)"); plt.legend()
plt.title("Stage 2: instruction tuning"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

if eval_pts:
    print(f"eval loss: {eval_pts[0][1]:.3f} (epoch {eval_pts[0][0]:.0f}) "
          f"-> {eval_pts[-1][1]:.3f} (epoch {eval_pts[-1][0]:.0f})")

## 10. Save the stage-2 adapter

Saving **before** the comparison below, because that section frees the model to make room for
loading the pristine base.

In [ ]:
import shutil

ADAPTER_DIR = Path(OUT_DIR)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Saved to {ADAPTER_DIR}  ({size_mb:.1f} MB)")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/finetuning-demo/stage2-instruct-lora")
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(ADAPTER_DIR, dest)
    print(f"Copied to {dest}")
except ImportError:
    print("Not running in Colab — adapter is on local disk only.")

## 11. The three-way comparison

The payoff. Same questions, three models:

| | what it is |
|---|---|
| **base** | `unsloth/Llama-3.2-1B`, untouched |
| **+ domain** | after stage 1 — the merged weights currently under the stage-2 adapter |
| **+ domain + instruct** | after stage 2 — the full pipeline |

**Memory trick:** the middle model comes free. `model.disable_adapter()` switches the stage-2
LoRA off in place, and what's left is precisely the merged stage-1 model. Only the pristine base
needs a separate load, and we free everything else first.

In [ ]:
@torch.no_grad()
def answer(m, record, max_new_tokens=160, use_template=True):
    """Generate a response. Greedy decoding so comparisons are deterministic."""
    m.eval()
    m.config.use_cache = True
    prompt = build_prompt(record) if use_template else record["instruction"]
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = m.generate(
        **ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,   # stop when the model says stop
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()


# Held-out instructions the model has never seen, plus the memorization probes from notebook 01.
# `subject` and `section` records share a fixed instruction string, so results are keyed by
# position rather than by text — otherwise two of them would collide into one row.
QUESTIONS = eval_records[:4] + [
    {"instruction": "How often must an H1 model be revalidated?", "input": ""},
    {"instruction": "What is the amber window?", "input": ""},
    {"instruction": "What is Meridian Trust's policy on time travel?", "input": ""},
]

results = []
for q in QUESTIONS:
    row = {"q": q, "expected": q.get("output"), "stage2": answer(model, q)}
    with model.disable_adapter():
        row["stage1"] = answer(model, q)
    results.append(row)

print(f"collected stage-1 and stage-2 answers for {len(results)} questions")

In [ ]:
# Free everything, then load the pristine base for the third column.
del trainer, model
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory after freeing: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
base_model.config.pad_token_id = tokenizer.pad_token_id

for row in results:
    row["base"] = answer(base_model, row["q"])

del base_model
gc.collect(); torch.cuda.empty_cache()
print("done")

In [ ]:
def show(text, limit=440):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[:limit] + " ..."


for row in results:
    q = row["q"]
    print("=" * 112)
    print(f"Q: {q['instruction']}")
    if q["input"].strip():
        print(f"   [input] {show(q['input'], 200)}")
    print()
    print(f"  [1] base                 : {show(row['base'])}\n")
    print(f"  [2] + domain (stage 1)   : {show(row['stage1'])}\n")
    print(f"  [3] + domain + instruct  : {show(row['stage2'])}\n")
    if row.get("expected"):
        print(f"  reference answer         : {show(row['expected'])}")
print("=" * 112)

### What to look for

- **Column 1 (base)** rambles. It has no idea what Meridian Trust is, and having never been
  instruction-tuned, it continues the prompt rather than answering it — often generating more
  `### Instruction:` blocks.
- **Column 2 (+ domain)** uses the right vocabulary and register but still *continues* rather
  than answers, and doesn't stop cleanly. This is what domain adaptation alone buys you.
- **Column 3 (+ instruct)** answers the question, in the domain's language, and **stops** —
  because EOS was in the training targets.
- The last question is deliberately unanswerable. Watch whether column 3 declines. With only 13
  abstention examples in training it may not, and that's an honest result worth seeing.

In [ ]:
# One more angle: how long does each stage's output run before it stops?
print("base   st1   st2   question")
for row in results:
    lens = [len(tokenizer(row[k], add_special_tokens=False)["input_ids"])
            for k in ("base", "stage1", "stage2")]
    print(f"{lens[0]:>4}  {lens[1]:>4}  {lens[2]:>4}   {row['q']['instruction'][:66]}")
print("\n(tokens generated before EOS or the 160-token cap)")
print("Stage-2 numbers well under 160 mean the model learned to stop on its own.")

## 12. What you built

```
unsloth/Llama-3.2-1B  (base, no instruction tuning)
    └─ stage 1: LoRA on 30k tokens of raw domain text     ~11M params
         └─ merged into base weights
              └─ stage 2: LoRA on 295 instruction pairs   ~11M params
```

Two adapters, ~45 MB each, on top of a 1.24B-parameter model, trained end to end on a free T4.

### The four bugs this pipeline avoids

1. Loading an adapter with `AutoModelForCausalLM.from_pretrained(adapter_dir)` — silently gives
   you the base model. We used `PeftModel.from_pretrained` and asserted the weights changed.
2. `labels = input_ids.copy()` with `padding="max_length"` — trains on padding. We packed in
   stage 1 and masked with `-100` in stage 2.
3. Supervising the prompt — teaches the model to generate instructions. We masked the prompt span.
4. No EOS on targets — the model never stops. We appended it and measured the generation lengths.

### Honest limitations

- **295 training examples is very small.** Real instruction tuning uses 10³–10⁶. Expect correct
  *form* — answering, stopping, using the right register — more than reliable *content*.
- **The model will still be confidently wrong.** Nothing here grounds it in retrieved text, and
  nothing checks its claims.
- **Abstention is undertrained** at ~4% of the set. The corpus itself argues for ~10%.
- **Both eval sets are small**, so treat the metrics as directional.

### Where to go next

| | |
|---|---|
| **Scale the data** | The single highest-return change. Both stages are data-starved. |
| **TRL `SFTTrainer`** | Does the masking and packing for you via `DataCollatorForCompletionOnlyLM`. Worth using once you understand what it's doing. |
| **QLoRA** | 4-bit base weights via `bitsandbytes` — needed once the model is 7B+. Note you can't cleanly merge into quantized weights, which is why this repo didn't use it. |
| **Unsloth** | ~2x faster with lower memory for exactly this workload. |
| **Preference tuning** | DPO / ORPO on preference pairs — the stage after this one. |
| **GGUF export** | `llama.cpp` conversion to run the merged model locally. |

See `docs/concepts.md` for the reasoning behind each decision here.